# Streaming Algorithms

**Core constraint**: Process data in one pass with O(1) or O(sublinear) memory.

Covers:
1. Welford's online mean and variance
2. Reservoir sampling
3. Count-Min Sketch (frequency estimation)
4. HyperLogLog (cardinality estimation)
5. Exponentially Weighted Moving Average (EWMA)
6. Sliding window statistics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, deque
import hashlib

np.random.seed(42)

## 1. Welford's Online Algorithm

Compute running mean and variance in a single pass with O(1) memory.

Naive approach $\text{Var} = E[X^2] - (E[X])^2$ suffers from catastrophic cancellation.

**Welford's update** (numerically stable):
- $\delta = x_n - \bar{x}_{n-1}$
- $\bar{x}_n = \bar{x}_{n-1} + \delta / n$
- $M_{2,n} = M_{2,n-1} + \delta \cdot (x_n - \bar{x}_n)$
- $\sigma^2_n = M_{2,n} / n$ (population) or $M_{2,n} / (n-1)$ (sample)

In [ ]:
class WelfordStats:
    """Online mean and variance using Welford's algorithm."""
    
    def __init__(self):
        self.n = 0
        self.mean = 0.0
        self.M2 = 0.0  # sum of squared deviations from current mean
    
    def update(self, x):
        self.n += 1
        delta = x - self.mean
        self.mean += delta / self.n
        delta2 = x - self.mean  # note: uses updated mean
        self.M2 += delta * delta2
    
    def variance(self, ddof=0):
        """ddof=0 for population variance, ddof=1 for sample variance."""
        if self.n < 2:
            return 0.0
        return self.M2 / (self.n - ddof)
    
    def std(self, ddof=0):
        return np.sqrt(self.variance(ddof))


# Demo: compare to numpy
data = np.random.randn(10000) * 5 + 100  # high mean to test numerical stability

ws = WelfordStats()
running_means = []
running_vars = []
for x in data:
    ws.update(x)
    running_means.append(ws.mean)
    running_vars.append(ws.variance(ddof=1))

print(f"Welford  mean={ws.mean:.6f}, var={ws.variance(ddof=1):.6f}")
print(f"NumPy    mean={np.mean(data):.6f}, var={np.var(data, ddof=1):.6f}")
print(f"Diff     mean={abs(ws.mean - np.mean(data)):.2e}, var={abs(ws.variance(ddof=1) - np.var(data, ddof=1)):.2e}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(running_means, linewidth=0.5)
axes[0].axhline(np.mean(data), color='red', linestyle='--', label='true mean')
axes[0].set_xlabel('Number of samples')
axes[0].set_ylabel('Running mean')
axes[0].set_title("Welford's Running Mean")
axes[0].legend()

axes[1].plot(running_vars, linewidth=0.5)
axes[1].axhline(np.var(data, ddof=1), color='red', linestyle='--', label='true variance')
axes[1].set_xlabel('Number of samples')
axes[1].set_ylabel('Running variance')
axes[1].set_title("Welford's Running Variance")
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Reservoir Sampling

**Problem**: Maintain a uniform random sample of size k from a stream of unknown length.

**Algorithm**: For the i-th element (1-indexed):
- If i <= k: add to reservoir
- If i > k: with probability k/i, replace a random element in the reservoir

**Proof of uniformity**: After processing n elements, each element has probability k/n of being in the reservoir.
- Base case: After k elements, each has probability k/k = 1.
- Inductive step: Assume after n-1 elements, each has prob k/(n-1). For element n:
  - P[element n in reservoir] = k/n (by algorithm)
  - P[old element survives] = P[was in reservoir] * P[not replaced] = k/(n-1) * (1 - k/n * 1/k) = k/(n-1) * (n-1)/n = k/n

In [ ]:
def reservoir_sampling(stream, k):
    """
    Reservoir sampling: maintain uniform sample of size k from stream.
    """
    reservoir = []
    for i, item in enumerate(stream):
        if i < k:
            reservoir.append(item)
        else:
            # Replace element j with probability k/(i+1)
            j = np.random.randint(0, i + 1)
            if j < k:
                reservoir[j] = item
    return reservoir


# Verify uniformity: sample from stream of integers, check each has equal probability
stream_size = 1000
k = 10
n_trials = 50000

counts = np.zeros(stream_size)
for _ in range(n_trials):
    sample = reservoir_sampling(range(stream_size), k)
    for s in sample:
        counts[s] += 1

expected = n_trials * k / stream_size

print(f"Expected count per element: {expected:.1f}")
print(f"Actual mean: {counts.mean():.1f}, std: {counts.std():.2f}")
print(f"Min: {counts.min():.0f}, Max: {counts.max():.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Sampling frequency for each stream position
axes[0].plot(counts / n_trials, linewidth=0.5, alpha=0.7)
axes[0].axhline(k / stream_size, color='red', linestyle='--', label=f'expected = {k/stream_size:.3f}')
axes[0].set_xlabel('Stream position')
axes[0].set_ylabel('Empirical selection probability')
axes[0].set_title('Reservoir Sampling Uniformity')
axes[0].legend()

# Histogram of counts
axes[1].hist(counts, bins=30, edgecolor='black', alpha=0.7)
axes[1].axvline(expected, color='red', linestyle='--', label=f'expected = {expected:.0f}')
axes[1].set_xlabel('Times selected')
axes[1].set_ylabel('Number of elements')
axes[1].set_title('Distribution of Selection Counts')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Count-Min Sketch

**Problem**: Estimate frequency of items in a stream using sub-linear memory.

**Structure**: A 2D array of counters with `depth` rows and `width` columns.
- Each row uses a different hash function.
- **Update**: For item x, increment `table[i][h_i(x)]` for all rows i.
- **Query**: Return `min_i(table[i][h_i(x)])` -- minimum across rows.

**Error bound**: With probability $\ge 1 - \delta$, the estimate $\hat{f}(x) \le f(x) + \epsilon N$
where $N$ is total count, $\text{width} = \lceil e/\epsilon \rceil$, $\text{depth} = \lceil \ln(1/\delta) \rceil$.

Key property: **never underestimates** (only overestimates due to hash collisions).

In [ ]:
class CountMinSketch:
    """Count-Min Sketch for frequency estimation."""
    
    def __init__(self, width, depth):
        self.width = width
        self.depth = depth
        self.table = np.zeros((depth, width), dtype=np.int64)
        # Use different hash seeds for each row
        self.seeds = [np.random.randint(0, 2**31) for _ in range(depth)]
    
    def _hash(self, item, row):
        """Hash item to a column index for given row."""
        h = hashlib.md5(f"{self.seeds[row]}_{item}".encode()).hexdigest()
        return int(h, 16) % self.width
    
    def update(self, item, count=1):
        """Increment counters for item."""
        for row in range(self.depth):
            col = self._hash(item, row)
            self.table[row, col] += count
    
    def query(self, item):
        """Estimate frequency of item (never underestimates)."""
        estimates = []
        for row in range(self.depth):
            col = self._hash(item, row)
            estimates.append(self.table[row, col])
        return min(estimates)


# Generate a stream with Zipf distribution (some items very frequent)
n_items = 10000
stream_length = 100000
# Zipf: item i has probability proportional to 1/(i+1)
probs = 1.0 / np.arange(1, n_items + 1)
probs /= probs.sum()
stream = np.random.choice(n_items, size=stream_length, p=probs)

# Exact counts
exact_counts = defaultdict(int)
for item in stream:
    exact_counts[item] += 1

# Build Count-Min Sketch
cms = CountMinSketch(width=1000, depth=5)
for item in stream:
    cms.update(item)

# Check accuracy on top-100 items
top_items = sorted(exact_counts.keys(), key=lambda x: exact_counts[x], reverse=True)[:100]
errors = []
for item in top_items:
    est = cms.query(item)
    exact = exact_counts[item]
    errors.append(est - exact)
    if item in top_items[:5]:
        print(f"Item {item}: exact={exact}, estimate={est}, error={est-exact}")

print(f"\nTop-100 items: mean overestimate = {np.mean(errors):.1f}, max = {np.max(errors)}")

In [ ]:
# Benchmark: error vs width and depth
widths = [50, 100, 200, 500, 1000, 2000]
depths = [2, 3, 5, 7]

error_matrix = np.zeros((len(depths), len(widths)))

for i, depth in enumerate(depths):
    for j, width in enumerate(widths):
        cms_test = CountMinSketch(width=width, depth=depth)
        for item in stream:
            cms_test.update(item)
        
        # Mean absolute error on all items
        errs = []
        for item in exact_counts:
            errs.append(cms_test.query(item) - exact_counts[item])
        error_matrix[i, j] = np.mean(errs)

fig, ax = plt.subplots(figsize=(8, 5))
for i, depth in enumerate(depths):
    ax.plot(widths, error_matrix[i], marker='o', label=f'depth={depth}')
ax.set_xlabel('Width')
ax.set_ylabel('Mean overestimate')
ax.set_title('Count-Min Sketch: Error vs Width/Depth')
ax.legend()
ax.set_xscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. HyperLogLog (Cardinality Estimation)

**Problem**: Estimate the number of distinct elements in a stream.

**Key insight**: If you hash elements uniformly, the maximum number of leading zeros in the hash values is approximately $\log_2(n)$ where $n$ is the number of distinct elements.

**Algorithm**:
1. Hash each element to a binary string
2. Use first $p$ bits to choose one of $m = 2^p$ registers
3. Remaining bits: count leading zeros + 1, store max in the register
4. Estimate: harmonic mean of $2^{\text{register}_i}$, scaled by constant

Memory: O(m) registers, typically m = 1024-16384 (uses ~1-12 KB for billions of items).

In [ ]:
class HyperLogLog:
    """HyperLogLog cardinality estimator."""
    
    def __init__(self, p=10):
        """
        p: precision parameter. Uses m = 2^p registers.
        Standard error ~ 1.04 / sqrt(m)
        """
        self.p = p
        self.m = 1 << p  # 2^p registers
        self.registers = np.zeros(self.m, dtype=np.int8)
        # Alpha constant for bias correction
        if self.m >= 128:
            self.alpha = 0.7213 / (1 + 1.079 / self.m)
        elif self.m == 64:
            self.alpha = 0.709
        elif self.m == 32:
            self.alpha = 0.697
        else:
            self.alpha = 0.673
    
    def _hash(self, item):
        """Hash item to 64-bit integer."""
        h = hashlib.sha256(str(item).encode()).hexdigest()
        return int(h[:16], 16)  # 64-bit hash
    
    def _leading_zeros(self, value, max_bits=64):
        """Count leading zeros + 1 in binary representation."""
        if value == 0:
            return max_bits
        count = 1
        while (value & 1) == 0 and count < max_bits:
            count += 1
            value >>= 1
        return count
    
    def add(self, item):
        h = self._hash(item)
        # First p bits determine the register
        register_idx = h & (self.m - 1)
        # Remaining bits: count leading zeros
        remaining = h >> self.p
        lz = self._leading_zeros(remaining, 64 - self.p)
        self.registers[register_idx] = max(self.registers[register_idx], lz)
    
    def count(self):
        """Estimate cardinality."""
        # Harmonic mean of 2^register values
        indicator = np.sum(2.0 ** (-self.registers.astype(float)))
        estimate = self.alpha * self.m * self.m / indicator
        
        # Small range correction
        if estimate <= 2.5 * self.m:
            zeros = np.sum(self.registers == 0)
            if zeros > 0:
                estimate = self.m * np.log(self.m / zeros)
        
        return int(estimate)


# Demo: estimate cardinality of streams of various sizes
true_sizes = [100, 500, 1000, 5000, 10000, 50000, 100000]
estimates = []

for n in true_sizes:
    hll = HyperLogLog(p=10)  # 1024 registers
    for i in range(n):
        hll.add(i)
    est = hll.count()
    estimates.append(est)
    err_pct = abs(est - n) / n * 100
    print(f"True={n:>7d}, Estimate={est:>7d}, Error={err_pct:.1f}%")

print(f"\nTheoretical std error: {1.04/np.sqrt(1024)*100:.1f}% (for p=10, m=1024)")

## 5. Exponentially Weighted Moving Average (EWMA)

$\text{EWMA}_t = \alpha \cdot x_t + (1 - \alpha) \cdot \text{EWMA}_{t-1}$

- $\alpha$ close to 1: more weight on recent data (responsive, noisy)
- $\alpha$ close to 0: more weight on history (smooth, laggy)
- Effective window: approximately $1/\alpha$ observations

In [ ]:
class EWMA:
    """Exponentially Weighted Moving Average."""
    
    def __init__(self, alpha=0.1):
        self.alpha = alpha
        self.value = None
    
    def update(self, x):
        if self.value is None:
            self.value = x
        else:
            self.value = self.alpha * x + (1 - self.alpha) * self.value
        return self.value


# Demo with a noisy signal that has a level shift
n = 500
signal = np.concatenate([
    np.random.randn(200) + 0,
    np.random.randn(150) + 3,
    np.random.randn(150) + 1
])

alphas = [0.02, 0.1, 0.3]
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(signal, alpha=0.3, label='raw signal', linewidth=0.5)

for alpha in alphas:
    ewma = EWMA(alpha=alpha)
    smoothed = [ewma.update(x) for x in signal]
    ax.plot(smoothed, label=f'EWMA (alpha={alpha})', linewidth=2)

ax.set_xlabel('Time')
ax.set_ylabel('Value')
ax.set_title('EWMA with Different Smoothing Parameters')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Sliding Window Statistics

Maintain statistics over the last W elements using a deque.

For mean/sum: O(1) update by subtracting the element falling out of the window.
For min/max: use a monotonic deque for O(1) amortized updates.

In [ ]:
class SlidingWindowStats:
    """Sliding window mean, variance, min, max."""
    
    def __init__(self, window_size):
        self.W = window_size
        self.buffer = deque(maxlen=window_size)
        self.sum_ = 0.0
        self.sum_sq = 0.0
        # Monotonic deques for min/max
        self.min_deque = deque()  # stores (value, index) in increasing order
        self.max_deque = deque()  # stores (value, index) in decreasing order
        self.idx = 0
    
    def update(self, x):
        # Remove old element if window is full
        if len(self.buffer) == self.W:
            old = self.buffer[0]
            self.sum_ -= old
            self.sum_sq -= old * old
        
        self.buffer.append(x)
        self.sum_ += x
        self.sum_sq += x * x
        
        # Update min deque (monotonic increasing)
        while self.min_deque and self.min_deque[-1][0] >= x:
            self.min_deque.pop()
        self.min_deque.append((x, self.idx))
        # Remove elements outside window
        while self.min_deque[0][1] <= self.idx - self.W:
            self.min_deque.popleft()
        
        # Update max deque (monotonic decreasing)
        while self.max_deque and self.max_deque[-1][0] <= x:
            self.max_deque.pop()
        self.max_deque.append((x, self.idx))
        while self.max_deque[0][1] <= self.idx - self.W:
            self.max_deque.popleft()
        
        self.idx += 1
    
    def mean(self):
        n = len(self.buffer)
        return self.sum_ / n if n > 0 else 0.0
    
    def variance(self):
        n = len(self.buffer)
        if n < 2:
            return 0.0
        mean = self.sum_ / n
        return self.sum_sq / n - mean * mean
    
    def min(self):
        return self.min_deque[0][0] if self.min_deque else float('inf')
    
    def max(self):
        return self.max_deque[0][0] if self.max_deque else float('-inf')


# Demo
W = 50
sw = SlidingWindowStats(W)
data_stream = np.random.randn(500) + np.sin(np.linspace(0, 6 * np.pi, 500)) * 2

means, mins, maxs = [], [], []
for x in data_stream:
    sw.update(x)
    means.append(sw.mean())
    mins.append(sw.min())
    maxs.append(sw.max())

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(data_stream, alpha=0.3, linewidth=0.5, label='raw')
ax.plot(means, label=f'sliding mean (W={W})', linewidth=2)
ax.fill_between(range(len(data_stream)), mins, maxs, alpha=0.2, label='min-max band')
ax.set_xlabel('Time')
ax.set_ylabel('Value')
ax.set_title('Sliding Window Statistics')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Full Demo: Processing a Synthetic Data Stream

Apply all algorithms simultaneously to a single stream.

In [ ]:
# Synthetic stream: user IDs with Zipf distribution visiting a website
n_events = 50000
n_users = 5000
probs_users = 1.0 / np.arange(1, n_users + 1) ** 0.8
probs_users /= probs_users.sum()
user_stream = np.random.choice(n_users, size=n_events, p=probs_users)

# Also generate a numeric feature per event (e.g., latency)
latency_stream = np.abs(np.random.randn(n_events) * 50 + 100)

# Process with all algorithms
welford = WelfordStats()
reservoir = []
k_reservoir = 100
cms = CountMinSketch(width=500, depth=5)
hll = HyperLogLog(p=12)
ewma = EWMA(alpha=0.01)

# Track metrics over time
checkpoints = list(range(0, n_events, 1000))
hll_estimates = []
true_distinct = []
ewma_values = []
welford_means = []

seen_users = set()

for i in range(n_events):
    user = user_stream[i]
    latency = latency_stream[i]
    
    # Welford: track latency stats
    welford.update(latency)
    
    # Reservoir sampling
    if i < k_reservoir:
        reservoir.append(user)
    else:
        j = np.random.randint(0, i + 1)
        if j < k_reservoir:
            reservoir[j] = user
    
    # Count-Min Sketch: track user frequencies
    cms.update(user)
    
    # HyperLogLog: track distinct users
    hll.add(user)
    seen_users.add(user)
    
    # EWMA: track smoothed latency
    ewma.update(latency)
    
    if i in checkpoints or i == n_events - 1:
        hll_estimates.append(hll.count())
        true_distinct.append(len(seen_users))
        ewma_values.append(ewma.value)
        welford_means.append(welford.mean)

print(f"After {n_events} events:")
print(f"  Welford mean latency: {welford.mean:.2f} (true: {np.mean(latency_stream):.2f})")
print(f"  Welford std latency: {welford.std(ddof=1):.2f} (true: {np.std(latency_stream, ddof=1):.2f})")
print(f"  HLL distinct users: {hll.count()} (true: {len(seen_users)})")
print(f"  CMS freq of user 0: {cms.query(0)} (true: {np.sum(user_stream == 0)})")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# HLL estimates over time
axes[0, 0].plot(checkpoints[:len(true_distinct)], true_distinct, label='true distinct', linewidth=2)
axes[0, 0].plot(checkpoints[:len(hll_estimates)], hll_estimates, '--', label='HLL estimate', linewidth=2)
axes[0, 0].set_xlabel('Events processed')
axes[0, 0].set_ylabel('Distinct users')
axes[0, 0].set_title('HyperLogLog: Distinct Count Over Time')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# EWMA vs running mean
axes[0, 1].plot(checkpoints[:len(ewma_values)], ewma_values, label='EWMA', linewidth=2)
axes[0, 1].plot(checkpoints[:len(welford_means)], welford_means, '--', label='Welford mean', linewidth=2)
axes[0, 1].set_xlabel('Events processed')
axes[0, 1].set_ylabel('Latency')
axes[0, 1].set_title('EWMA vs Running Mean of Latency')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# CMS accuracy: top-50 users
exact_user_counts = defaultdict(int)
for u in user_stream:
    exact_user_counts[u] += 1
top50 = sorted(exact_user_counts.keys(), key=lambda x: exact_user_counts[x], reverse=True)[:50]
exact_freqs = [exact_user_counts[u] for u in top50]
cms_freqs = [cms.query(u) for u in top50]
axes[1, 0].scatter(exact_freqs, cms_freqs, alpha=0.6, s=20)
max_f = max(max(exact_freqs), max(cms_freqs))
axes[1, 0].plot([0, max_f], [0, max_f], 'r--', label='perfect')
axes[1, 0].set_xlabel('Exact frequency')
axes[1, 0].set_ylabel('CMS estimate')
axes[1, 0].set_title('Count-Min Sketch Accuracy (top-50 users)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Reservoir sample distribution
reservoir_counts = defaultdict(int)
for u in reservoir:
    reservoir_counts[u] += 1
axes[1, 1].bar(range(min(30, len(reservoir_counts))),
               [reservoir_counts[u] for u in sorted(reservoir_counts.keys())[:30]],
               alpha=0.7)
axes[1, 1].set_xlabel('User ID (first 30 in sample)')
axes[1, 1].set_ylabel('Count in reservoir')
axes[1, 1].set_title(f'Reservoir Sample (k={k_reservoir})')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Interview Questions & Answers

### "How to compute mean/variance on a stream?"
Use Welford's algorithm. O(1) memory, numerically stable. Key insight: track the sum of squared deviations from the *current* mean (M2), not from zero. Update rule avoids the catastrophic cancellation in the naive $E[X^2] - (E[X])^2$ formula.

### "Reservoir sampling -- prove it gives a uniform sample?"
Induction on stream length n. After k elements, each is in the reservoir with probability 1. For element i > k: P[selected] = k/i. For any old element: P[still in reservoir] = P[was in] * P[not kicked out] = k/(i-1) * (1 - 1/(i) * k/k * 1) -- simplifies to k/i after algebra. So all elements have equal probability k/n.

### "Approximate counting?"
- **Frequencies**: Count-Min Sketch. 2D hash table, min over rows. Never underestimates. Error bounded by epsilon*N with high probability.
- **Distinct count**: HyperLogLog. Uses max leading zeros in hash values. O(m) memory for error ~1.04/sqrt(m). In practice, 12 KB tracks billions of distinct elements.

## When These Matter in ML Pipelines

| Algorithm | ML Application |
|-----------|----------------|
| Welford's | Online feature normalization, monitoring model predictions |
| Reservoir sampling | Balanced batching from imbalanced streams, data collection |
| Count-Min Sketch | Feature frequency counting for hashing trick, rare event detection |
| HyperLogLog | Tracking unique users/items for cardinality features |
| EWMA | Monitoring training loss, detecting distribution shift |
| Sliding window | Real-time feature engineering, anomaly detection |